[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/surfaces/qem/qem.ipynb)

# Quadric Error Metrics

This example simplifies a mesh by collapsing one of its edges into a single vertex, placed where it disturbs the surface least. Every triangle of the mesh lies in a plane, and moving a vertex off the planes of its triangles costs the sum of its squared distances to them. Collapsing an edge merges its two endpoints into one vertex that must serve the triangles of both, and the best place for it is where that summed cost is least: on a sharp ridge it stays on the ridge, pulled towards the sharper end.

In projective geometric algebra a plane is a vector and a point an antivector, and their pairing, `plane & point`, is the point's signed distance from a unit plane. A plane's dyad, `plane * (plane & Point)`, is a map from points to planes whose value at a point is its squared distance, and the error quadric of a vertex is the sum of its triangles' dyads. Collapsing an edge adds the endpoints' quadrics, and the best vertex is the point whose polar plane is the plane at infinity, where the cost has no slope: one solve of the edge's quadric against `mv.w`.

In the matrix notation of Garland and Heckbert each plane reads as a vector $\mathbf p$ of its four coefficients, its quadric as $\mathbf p\mathbf p^{\mathsf T}$, and the best vertex as the solution of $Q\,\mathbf v = (0, 0, 0, 1)^{\mathsf T}$ with the last row of $Q$ replaced.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import numpy as np

from numga import concatenate
from examples.animation import save_figure
from examples.surfaces.qem import render
from examples.surfaces.qem.core import Plane, Point, Quadric, mv, ridge_patch

np.set_printoptions(precision=4, suppress=True)

## 1. A ridge, and its triangles' planes

The patch is six vertices and six triangles: a ridge from a sharp corner a to a gentler end b, two faces flanking the ridge edge, two steep faces at the corner and two shallow ones at the end. Each triangle's plane is the join of its three vertices.

In [ ]:
# The faces incident to each endpoint of the ridge edge (a, b), the two flanking faces in both; the faces
# that survive the collapse.
incident_a = np.array([0, 1, 2, 3])
incident_b = np.array([0, 1, 4, 5])
surviving = np.array([2, 3, 4, 5])

In [ ]:
vertices, faces = ridge_patch()                                                # [vertices] Point, [faces, 3]
face_planes = (vertices[faces[:, 0]] & vertices[faces[:, 1]] & vertices[faces[:, 2]]).normalized()   # [faces] Plane

In [ ]:
print("each face's plane against its three vertices:", np.abs((face_planes[:, None] & vertices[faces]).to_array()).max())

## 2. Collapsing the edge

Each endpoint's quadric is the sum of its triangles' plane dyads, and the edge's quadric their sum. The best vertex for the edge has the plane at infinity as its polar plane. In the figure, left and middle, each endpoint with its triangles and the region where its error stays small; right, the collapsed mesh around the best vertex, green, with the original mesh faint behind it. The two flanking faces vanish with the edge.

In [ ]:
def edge_collapse(
    planes_a: Plane,
    planes_b: Plane,
):
    """Contract a mesh edge (a, b) by summing incident face plane quadrics.

    planes_a and planes_b are the supporting planes of the mesh faces incident
    to the edge's two endpoint vertices, a and b. In a 2-manifold mesh, the
    faces flanking the edge contain both endpoints and appear in both sets.
    """
    # A unit plane measures a point's signed distance from it as `plane & point`; its dyad,
    # `plane * (plane & Point)`, a map from points to planes, gives the squared distance.
    # Summing the incident planes' dyads at each vertex gives its error quadric.
    quadric_a: Quadric = (planes_a * (planes_a & Point)).sum(axis=0)
    quadric_b: Quadric = (planes_b * (planes_b & Point)).sum(axis=0)
    # Contracting the edge adds the endpoints' quadrics.
    edge_quadric: Quadric = quadric_a + quadric_b
    # The optimal vertex has vanishing spatial gradient, so its polar plane is the plane at infinity.
    optimum: Point = edge_quadric.lstsq(mv.w).normalized()
    return quadric_a, quadric_b, edge_quadric, optimum


quadric_a, quadric_b, edge_quadric, optimum = edge_collapse(face_planes[incident_a], face_planes[incident_b])
collapsed = concatenate([optimum.broadcast_to((2,)), vertices[2:]])            # [vertices] Point
save_figure(render.draw_qem(vertices, faces, incident_a, incident_b, quadric_a, quadric_b, edge_quadric, optimum,
                            collapsed, surviving), "qem_mesh_simplification");

In [ ]:
endpoints = vertices[:2]
print("the best vertex:", render.euclidean(optimum))
print("its error, against the endpoints':", (edge_quadric(optimum) & optimum).to_array(), (edge_quadric(endpoints) & endpoints).to_array())

In [ ]:
# checks
# The best vertex's polar plane is the plane at infinity; it stays on the ridge, between its middle and
# the sharp corner, and its error undercuts both endpoints'.
assert edge_quadric(optimum).norm().to_array() < 1e-9
np.testing.assert_allclose(render.euclidean(optimum)[1], 0.0, atol=1e-12)
assert 0.0 < render.euclidean(optimum)[0] < render.euclidean(vertices[0])[0]
assert np.all((edge_quadric(optimum) & optimum).to_array() < (edge_quadric(endpoints) & endpoints).to_array())